# Stage 4 — Baseline + Merge + Experiments


## 1. Load data

In [1]:

import pandas as pd
from pathlib import Path

ROOT = Path("../")

emotion_path = ROOT / "datasets/emotion_annotated/metadata/pilot_video_emotion_features.csv"
manifest_path = ROOT / "exports/pilot/csv/pilot_manifest_export.csv"
detector_path = ROOT / "datasets/detector_processed/pilot_detector_scores.csv"

emotion_df = pd.read_csv(emotion_path)
manifest_df = pd.read_csv(manifest_path)
detector_df = pd.read_csv(detector_path)

print(len(emotion_df), len(manifest_df), len(detector_df))


196 200 196


## 2. Merge

In [2]:

# Drop duplicate columns from emotion_df before merging
emotion_drop = [c for c in emotion_df.columns if c in manifest_df.columns and c != "video_id"]
df = manifest_df.merge(emotion_df.drop(columns=emotion_drop), on="video_id")

# Only bring score columns from detector_df to avoid further duplicates
detector_cols = ["video_id", "detector_score", "detector_pred"]
df = df.merge(detector_df[detector_cols], on="video_id")

# Encode label: "fake" → 1, "real" → 0
df["label"] = (df["label"].str.lower() == "fake").astype(int)

print("Shape:", df.shape)
print("Label distribution:\n", df["label"].value_counts())
df.head()


Shape: (196, 73)
Label distribution:
 label
0    100
1     96
Name: count, dtype: int64


,video_id,path,relative_path,split,label,source_subset,manipulation_family,manipulation_type,identity,source_identity,...,mean_score_relief,mean_score_sadness,mean_score_sexual_lust,mean_score_shame,mean_score_sourness,mean_score_teasing,mean_score_thankfulness_gratitude,mean_score_triumph,detector_score,detector_pred
0,YouTube-real__00246__bc0e9215,videos/real/YouTube-real/00246.mp4,YouTube-real/00246.mp4,train,0,YouTube-real,NaN,YouTube-real,NaN,NaN,...,-0.174653,0.083933,2.062548,0.043499,-0.680376,0.464667,0.715369,-1.077735,0.270464,0
1,Celeb-real__id19_0003__02a18478,videos/real/Celeb-real/id19_0003.mp4,Celeb-real/id19_0003.mp4,test,0,Celeb-real,NaN,Celeb-real,id19,id19,...,0.609876,-0.438989,1.791388,-0.528835,-0.732823,1.037598,3.180103,0.175943,0.435596,0
2,Celeb-real__id5_0001__7446e574,videos/real/Celeb-real/id5_0001.mp4,Celeb-real/id5_0001.mp4,train,0,Celeb-real,NaN,Celeb-real,id5,id5,...,1.041872,-0.458066,1.268688,-0.249847,-0.550585,0.728570,1.921226,-0.706379,0.918671,1
3,Celeb-synthesis__FaceSwap__MobileFaceSwap__id2...,videos/fake/FaceSwap/id21_id1_0005.mp4,Celeb-synthesis/FaceSwap/MobileFaceSwap/id21_i...,val,1,Celeb-synthesis,FaceSwap,MobileFaceSwap,id21,id21,...,0.713487,-0.253131,2.422626,-0.400127,-0.577885,0.742537,2.218336,0.044266,0.665731,1
4,Celeb-synthesis__FaceReenact__MCNET__id35_id32...,videos/fake/FaceReenact/id35_id32_0006.mp4,Celeb-synthesis/FaceReenact/MCNET/id35_id32_00...,train,1,Celeb-synthesis,FaceReenact,MCNET,id35,id35,...,0.539897,-0.358464,1.602956,-0.440720,-0.947647,0.231478,0.761027,-1.354359,0.998819,1


## 3. Metrics (baseline)

In [3]:

import numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

y_true = df["label"].values
y_score = df["detector_score"].values
y_pred = (y_score >= 0.5).astype(int)

auc = roc_auc_score(y_true, y_score)
acc = accuracy_score(y_true, y_pred)
f1  = f1_score(y_true, y_pred)

# Bootstrap 95% CI for AUC
rng = np.random.default_rng(42)
boot_aucs = [
    roc_auc_score(y_true[idx := rng.integers(0, len(y_true), len(y_true))], y_score[idx])
    for _ in range(2000)
]
auc_lo, auc_hi = np.percentile(boot_aucs, [2.5, 97.5])

print(f"AUC : {auc:.4f}  (95% CI {auc_lo:.4f}–{auc_hi:.4f})")
print(f"ACC : {acc:.4f}")
print(f"F1  : {f1:.4f}")


AUC : 0.5396  (95% CI 0.4604–0.6172)
ACC : 0.5306
F1  : 0.6198


## 4. Emotion analysis

In [4]:

df["arousal_bin"] = pd.qcut(df["mean_arousal"], 3, labels=["low", "mid", "high"])

rng = np.random.default_rng(42)

print(f"{'Bin':<6}  {'AUC':>6}  {'95% CI':>18}  {'n':>4}")
print("-" * 42)
for g in ["low", "mid", "high"]:
    sub = df[df["arousal_bin"] == g]
    yt, ys = sub["label"].values, sub["detector_score"].values
    auc = roc_auc_score(yt, ys)
    boot = [
        roc_auc_score(yt[idx := rng.integers(0, len(yt), len(yt))], ys[idx])
        for _ in range(2000)
    ]
    lo, hi = np.percentile(boot, [2.5, 97.5])
    print(f"{str(g):<6}  {auc:.4f}  ({lo:.4f} – {hi:.4f})  {len(sub):>4}")


Bin        AUC              95% CI     n
------------------------------------------
low     0.5451  (0.3953 – 0.6889)    66
mid     0.5690  (0.4291 – 0.7114)    65
high    0.4478  (0.3002 – 0.5909)    65


## 5. Fusion model

In [5]:

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

features = [
    "detector_score",
    "mean_arousal",
    "std_arousal",
    "mean_valence",
    "emotion_entropy",
    "transition_rate",
]

X = df[features].fillna(0).values
y = df["label"].values

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fusion_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000)),
])

fusion_proba = cross_val_predict(fusion_pipe, X, y, cv=cv, method="predict_proba")[:, 1]
fusion_auc = roc_auc_score(y, fusion_proba)

rng = np.random.default_rng(42)
boot = [
    roc_auc_score(y[idx := rng.integers(0, len(y), len(y))], fusion_proba[idx])
    for _ in range(2000)
]
lo, hi = np.percentile(boot, [2.5, 97.5])

print(f"Fusion AUC (5-fold CV): {fusion_auc:.4f}  (95% CI {lo:.4f}–{hi:.4f})")


Fusion AUC (5-fold CV): 0.6076  (95% CI 0.5299–0.6877)


## 6. Compare

In [6]:

# ── Detector baseline (no model needed — direct scores) ──────────────────────
baseline_auc = roc_auc_score(y, df["detector_score"].values)

# ── Emotion-only (5-fold CV, same pipeline) ───────────────────────────────────
emotion_features = [
    "mean_arousal", "std_arousal", "mean_valence",
    "emotion_entropy", "transition_rate",
]
X_emo = df[emotion_features].fillna(0).values

emotion_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000)),
])
emotion_proba = cross_val_predict(emotion_pipe, X_emo, y, cv=cv, method="predict_proba")[:, 1]
emotion_auc = roc_auc_score(y, emotion_proba)

rng2 = np.random.default_rng(42)
def _boot_auc(yt, ys, n=2000):
    rng_ = np.random.default_rng(42)
    b = [roc_auc_score(yt[i := rng_.integers(0, len(yt), len(yt))], ys[i]) for _ in range(n)]
    return np.percentile(b, [2.5, 97.5])

b_lo, b_hi = _boot_auc(y, df["detector_score"].values)
e_lo, e_hi = _boot_auc(y, emotion_proba)
f_lo, f_hi = lo, hi  # from fusion cell above

print(f"{'Model':<20}  {'AUC':>6}  {'95% CI':>18}")
print("-" * 50)
print(f"{'Detector only':<20}  {baseline_auc:.4f}  ({b_lo:.4f} – {b_hi:.4f})")
print(f"{'Emotion only (CV)':<20}  {emotion_auc:.4f}  ({e_lo:.4f} – {e_hi:.4f})")
print(f"{'Fusion (CV)':<20}  {fusion_auc:.4f}  ({f_lo:.4f} – {f_hi:.4f})")


Model                    AUC              95% CI
--------------------------------------------------
Detector only         0.5396  (0.4604 – 0.6172)
Emotion only (CV)     0.6190  (0.5446 – 0.6997)
Fusion (CV)           0.6076  (0.5299 – 0.6877)
